# Implement ROPE from Scratch

### Problem Statement
Self-attention is position-agnostic. Rotary positional embeddings (RoPE) inject position by rotating the query and the key. Each pair of features is turned by an angle that depends on the token position and a fixed frequency. Values are not rotated.

Implement it from scratch in PyTorch. Do not use a positional-encoding module from Fairseq or Hugging Face.

---

### Requirements

1. **Rotary module**
   - `Rotary(dim, base=10000)`. `dim` is even.
   - For `i = 0, 1, ..., dim/2 - 1`, the frequency is `ω_i = 1 / base**(2i/dim)`. Feature `i` and feature `i + dim/2` share that frequency.
   - `forward(x, seq_dim=2)` reads the sequence length from `x` of shape `(batch, heads, seq, dim)`.
   - Return `cos` and `sin` with shape `(1, 1, seq, dim)`, so they broadcast over the batch and the heads. Duplicate each angle across the two halves: `[ω, ω]`.
   - Register the cached `cos` and `sin` tensors as buffers. Rebuild the cache when the sequence length or the device changes.

2. **Rotation helpers**
   - `rotate_half(x)` splits the last dimension into `x1` and `x2` and returns `cat(-x2, x1)`.
   - `apply_rotary_pos_emb(q, k, cos, sin)` returns `(q * cos) + (rotate_half(q) * sin)` and the same transformation of `k`.

3. **Check the result**
   - Build `Rotary(64)` and rotate a query and a key of shape `(2, 4, 8, 64)`.
   - Both outputs keep that shape, and each vector keeps its norm.

---

### Constraints

- Use only PyTorch.
- `dim` is even.
- Rotate queries and keys only.

---

<details>
  <summary>💡 Hint</summary>

  - `inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))`. `arange(0, dim, 2)` is `2i`.
  - `freqs = torch.einsum("i,j->ij", positions, inv_freq)`, then `emb = cat((freqs, freqs), dim=-1)`.
  - Cache `emb.cos()[None, None, :, :]` and `emb.sin()[None, None, :, :]` with `register_buffer`.
  - `rotate_half` returns `cat(-x2, x1)`. Apply `(x * cos) + (rotate_half(x) * sin)`.

</details>

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Rotary(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim must be even")
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.register_buffer("cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("sin_cached", torch.empty(0), persistent=False)
        self._seq_len = None

    def forward(self, x, seq_dim=2):
        seq_len = x.shape[seq_dim]
        cached = self.cos_cached
        stale = (
            self._seq_len != seq_len
            or cached.numel() == 0
            or cached.device != x.device
            or cached.dtype != x.dtype
        )
        if stale:
            inv_freq = self.inv_freq.to(device=x.device)
            positions = torch.arange(seq_len, device=x.device, dtype=inv_freq.dtype)
            freqs = torch.einsum("i,j->ij", positions, inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            # (1, 1, seq, dim) broadcasts over (batch, heads, seq, dim)
            self.cos_cached = emb.cos().to(dtype=x.dtype)[None, None, :, :]
            self.sin_cached = emb.sin().to(dtype=x.dtype)[None, None, :, :]
            self._seq_len = seq_len
        return self.cos_cached, self.sin_cached


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

In [1]:
torch.manual_seed(0)
batch, n_heads, seq_len, dim = 2, 4, 8, 64

q = torch.randn(batch, n_heads, seq_len, dim)
k = torch.randn(batch, n_heads, seq_len, dim)

rope = Rotary(dim)
cos, sin = rope(q)
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)

print(tuple(q_rot.shape), tuple(k_rot.shape))
assert cos.shape == (1, 1, seq_len, dim)
assert sin.shape == (1, 1, seq_len, dim)
assert q_rot.shape == q.shape and k_rot.shape == k.shape
assert torch.allclose(q_rot.norm(dim=-1), q.norm(dim=-1), atol=1e-4)
assert torch.allclose(k_rot.norm(dim=-1), k.norm(dim=-1), atol=1e-4)

(2, 4, 8, 64) (2, 4, 8, 64)
